In [1]:
from pathlib import Path

messages_dir = Path.cwd().resolve().parent / 'tests' / 'messages'
message_files = sorted(messages_dir.glob('msg*.txt'))
messages_list = []
globals().update(
    {
        file_path.stem: file_path.read_text(encoding='utf-8')
        for file_path in message_files
    }
)

for file_path in message_files:
    messages_list.append(file_path.stem)



for file_path in message_files:
    print(f"Loaded {file_path.name} into variable {file_path.stem}")


Loaded msg1.txt into variable msg1
Loaded msg2.txt into variable msg2
Loaded msg3.txt into variable msg3
Loaded msg4.txt into variable msg4
Loaded msg5.txt into variable msg5
Loaded msg6.txt into variable msg6
Loaded msg7.txt into variable msg7
Loaded msg8.txt into variable msg8


In [2]:
print(messages_list)

['msg1', 'msg2', 'msg3', 'msg4', 'msg5', 'msg6', 'msg7', 'msg8']


In [3]:
import sys,os
sys.path.append(os.path.abspath('..'))

sys.path.append(os.path.abspath('.'))

In [4]:
from tools.extractJSON import extract_expense_test,extract_info_meter_sheet_test,extract_info_electronic_sales_sheet_test,rewrite_report_for_llm

In [5]:
from expected.msg1 import fuel
print(fuel)

{'date': '22/06/2026', 'pumps': {'PMS 1': {'opening': 1709211.595, 'closing': 679761.213}, 'PMS 2': {'opening': 1396086.013, 'closing': 348384.545}, 'AGO 1': {'opening': 857618.798, 'closing': 549370.32}, 'AGO 2': {'opening': 449227.617, 'closing': 294349.184}}, 'rtt': {'PMS': 9.861, 'AGO': 8.812}}


In [6]:
import importlib
expected_outputs = {}

In [7]:
for msg in messages_list:
    module = importlib.import_module(f"expected.{msg}")
    fuel = module.fuel
    electronic_sales = module.electronic_sales
    expenses = module.expenses
    expected_outputs[msg] = {'fuel': fuel, 'electronic_sales': electronic_sales, 'expenses': expenses}

In [26]:
def evalaute_fuel(output:dict, expected: dict):
    expected_extracted_keys = set(expected.keys()).union(set(expected["pumps"].keys())).union(set(expected["rtt"].keys()))
    expected_meter_keys = set()
    nozzle_names = list(expected["pumps"].keys())
    for nozzle in nozzle_names:
        name = f"{nozzle}:opening"
        name1 = f"{nozzle}:closing"
        expected_meter_keys.add(name)
        expected_meter_keys.add(name1)
    total = len(expected_extracted_keys) + len(expected_meter_keys)
    found = 0
    for key in output.keys():
        if key in expected_extracted_keys:
            found +=1

    try:
        pump_keys = set(output["pumps"].keys())
        for pump_key in pump_keys:
            name = f"{pump_key}:opening"
            name1 = f"{pump_key}:closing"
            if name in expected_meter_keys:
                found += 1
            if name1 in expected_meter_keys:
                found += 1
            if pump_key in expected_extracted_keys:
                found += 1
    except KeyError:
        pass

    #checking rtts
    try:
        rtt_keys = set(output["rtt"].keys())
        for rtt_key in rtt_keys:
            if rtt_key in expected_extracted_keys:
                found += 1
    except KeyError:
        pass
    recall = f"Recall: {found}/{total} = {found/total:.2%}"

    expected_accurate = 11
    actual_accurate = 0
    #i expect 11 values to be accurate
    #checkign for date
    if output["date"] == expected["date"]:
        actual_accurate += 1
    #checking rtt
    try:
        rtt_keys = set(output["rtt"].keys())
        for rtt_key in rtt_keys:
            if output["rtt"][rtt_key] == expected["rtt"][rtt_key]:
                actual_accurate += 1
    except KeyError:
        pass
    #checking meter readings
    try:
        nozzle_names = list(expected["pumps"].keys())
        for nozzle in nozzle_names:
            try:
                if output["pumps"][nozzle]["opening"] == expected["pumps"][nozzle]["opening"]:
                    actual_accurate += 1
            except KeyError:
                pass
            try: 
                if output["pumps"][nozzle]["closing"] == expected["pumps"][nozzle]["closing"]:
                    actual_accurate += 1
            except KeyError:
                pass
    except KeyError:
        pass
    accuracy = f"Accuracy: {actual_accurate}/{expected_accurate} = {actual_accurate/expected_accurate:.2%}"
    
    return recall, accuracy


In [27]:
print(evalaute_fuel(expected_outputs["msg1"]["fuel"], expected_outputs["msg1"]["fuel"]))

('Recall: 17/17 = 100.00%', 'Accuracy: 11/11 = 100.00%')


In [10]:
def evalaute_electronic_sales(output: dict, expected: dict):
    expected_date = expected["electronic_sales"].get("date")
    expected_values = expected["electronic_sales"].get("electronic_sales", {})

    output_section = output
    output_date = output_section.get("date")
    output_values = output_section.get("electronic_sales", {}) if isinstance(output_section, dict) else {}

    total = len(expected_values) + 1
    found = 0
    accurate = 0

    if output_date == expected_date:
        found += 1
        accurate += 1
    elif output_date is not None:
        found += 1

    for key, expected_amount in expected_values.items():
        if key in output_values:
            found += 1
            if output_values[key] == expected_amount:
                accurate += 1

    recall = f"Recall: {found}/{total} = {found/total:.2%}"
    accuracy = f"Accuracy: {accurate}/{total} = {accurate/total:.2%}"
    return recall, accuracy


In [11]:
print(expected_outputs["msg1"]["electronic_sales"])

{'date': '22/06/2026', 'electronic_sales': {'MOMOPAY': None, 'AIRTEL': None, 'VISA CARD': 130000, 'RUBIS CARD': 31000, 'RUBIS APP': None}}


In [12]:
def evalaute_expenses(output: dict, expected: dict):
    expected_date = expected.get("date")
    expected_items = expected.get("expenses", [])
    expected_by_name = {item["expense_name"]: item["amount"] for item in expected_items}

    output_section = output
    output_date = output_section.get("date")
    output_items = output_section.get("expenses", []) if isinstance(output_section, dict) else []

    output_by_name = {
        item.get("expense_name"): item.get("amount")
        for item in output_items
        if isinstance(item, dict) and "expense_name" in item
    }

    total = len(expected_by_name)  + 1
    found = 0
    accurate = 0

    if output_date == expected_date:
        found += 1
        accurate += 1
    elif output_date is not None:
        found += 1

    for name, expected_amount in expected_by_name.items():
        if name in output_by_name:
            found += 1
            if output_by_name[name] == expected_amount:
                accurate += 1
        

    recall = f"Recall: {found}/{total} = {found/total:.2%}"
    accuracy = f"Accuracy: {accurate}/{total} = {accurate/total:.2%}"
    return recall, accuracy


In [13]:
print(expected_outputs["msg1"]["expenses"])

{'date': '22/06/2026', 'expenses': [{'expense_name': 'TP to bank', 'amount': 2000}, {'expense_name': 'At and survey', 'amount': 4000}]}


In [14]:
print(evalaute_expenses(expected_outputs["msg1"]['expenses'],expected_outputs["msg1"]['expenses']))


('Recall: 3/3 = 100.00%', 'Accuracy: 3/3 = 100.00%')


In [15]:
print(evalaute_electronic_sales(expected_outputs["msg1"],expected_outputs["msg1"]))

('Recall: 0/6 = 0.00%', 'Accuracy: 0/6 = 0.00%')


In [16]:
print(evalaute_expenses(extract_expense_test(msg1),expected_outputs["msg1"]['expenses']))

('Recall: 3/3 = 100.00%', 'Accuracy: 3/3 = 100.00%')


In [28]:
print(evalaute_fuel(extract_info_meter_sheet_test(msg1),expected_outputs["msg1"]["fuel"]))

('Recall: 14/17 = 82.35%', 'Accuracy: 1/11 = 9.09%')


In [ ]:
print(expected_outputs["msg1"]['expenses'])

{'date': '22/06/2026', 'expenses': [{'expense_name': 'TP to bank', 'amount': 2000}, {'expense_name': 'At and survey', 'amount': 4000}]}


In [29]:
extract_info_meter_sheet_test(msg1)

{'date': '22/06/2026',
 'pumps': {'PMS 1': {'opening': '1709211.595', 'closing': '679761.213'},
  'PMS 2': {'opening': '1396086.013', 'closing': '348384.545'},
  'PMS 3': {'opening': None, 'closing': None},
  'PMS 4': {'opening': None, 'closing': None},
  'AGO 2': {'opening': '449227.617', 'closing': '294349.184'},
  'AGO 3': {'opening': None, 'closing': None},
  'AGO 4': {'opening': None, 'closing': None}},
 'rtt': {'PMS': '9.861', 'AGO': '8.812'}}

In [23]:
expected_outputs["msg1"]['fuel']

{'date': '22/06/2026',
 'pumps': {'PMS 1': {'opening': 1709211.595, 'closing': 679761.213},
  'PMS 2': {'opening': 1396086.013, 'closing': 348384.545},
  'AGO 1': {'opening': 857618.798, 'closing': 549370.32},
  'AGO 2': {'opening': 449227.617, 'closing': 294349.184}},
 'rtt': {'PMS': 9.861, 'AGO': 8.812}}